In [1]:
import torch
from datasets import load_dataset
import pandas as pd
from torch import nn

/Users/vikaspandey/projects/genesis-lab/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ner_dataset = load_dataset("lhoestq/conll2003")
lm_dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

In [3]:
def clean_lm_data(lm_data):
    return [
        text.strip() for text in lm_data["text"] if text.strip() and not text.strip().startswith("=")
    ]

def build_vocab(lm_dataset, vocab_size=30000):
    word_freq = {}
    idx_to_word = {}
    word_to_idx = {}
    for line in lm_dataset["text"]:
        words = line.strip().split()
        for word in words:
            word = word.lower()
            word_freq[word] = word_freq.get(word, 0) + 1
    top_words = sorted(word_freq, key=word_freq.get, reverse=True)[:vocab_size]
    vocab = top_words + ["<UNK>", "<PAD>"]
    idx_to_word = {i: word for i, word in enumerate(vocab)}
    word_to_idx = {word: i for i, word in enumerate(vocab)}
    return vocab, idx_to_word, word_to_idx

def encode_sequence(tokens, word_to_idx):
    unk_idx = word_to_idx["<UNK>"]
    word_indices = [word_to_idx.get(token.lower(), unk_idx) for token in tokens]
    cap_indices = [1 if token[0].isupper() else 0 for token in tokens]
    return word_indices, cap_indices

def window_extraction(tokens, word_to_idx, labels, window_size=5):
    half_size = int(window_size/2)
    tokens_copy = ["<PAD>"]*half_size + tokens + ["<PAD>"]*half_size
    all_word_window = []
    all_cap_window = []
    output_labels = []
    unk_idx = word_to_idx.get("<UNK>")
    for i in range(0, len(tokens)):
        window = tokens_copy[i:(i + window_size)]
        all_word_window.append([word_to_idx.get(token.lower(), unk_idx) for token in window])
        all_cap_window.append([1 if token != "<PAD>" and token[0].isupper() else 0 for token in window])
        output_labels.append(labels[i])
    return all_word_window, all_cap_window, output_labels


In [7]:
class SharedEmbedding(nn.Module):
    def __init__(self, vocab_size, wsz, num_cap_features=2, cap_emb_dim=2):
        super().__init__()
        self.word_emb = nn.Embedding(vocab_size, wsz)
        self.cap_emb = nn.Embedding(num_cap_features, cap_emb_dim)
        nn.init.uniform_(self.word_emb.weight, -0.01, 0.01)
        nn.init.uniform_(self.cap_emb.weight, -0.01, 0.01)
    
    def forward(self, word_indices, cap_indices):
        word_embs = self.word_emb(word_indices)   # (B, ksz, wsz)
        cap_embs = self.cap_emb(cap_indices)      # (B, ksz, 2)
        features = torch.cat([word_embs, cap_embs], dim=-1)  # (B, ksz, wsz+2)
        return features

In [8]:
class WindowClassifier(nn.Module):
    def __init__(self, num_classes, ksz, wsz, cap_emb_dim, hidden_dim=None):
        super().__init__()
        self.num_classes = num_classes
        self.ksz = ksz
        self.wsz = wsz
        self.cap_emb_dim = cap_emb_dim
        self.hidden_dim = hidden_dim
        if hidden_dim is None:
            self.hidden = nn.Linear(ksz*(wsz + cap_emb_dim), hidden_dim)
            self.output = nn.Linear(hidden_dim, num_classes)
        else:
            self.output = nn.Linear(ksz*(wsz + cap_emb_dim), num_classes)
        for layer in [self.hidden, self.output] if hidden_dim else [self.output]:
            nn.init.uniform_(layer.weight, -0.05, 0.05)
            nn.init.uniform_(layer.bias)
    
    def forward(self, features):
        e = features.reshape(-1, self.ksz*(self.wsz+self.cap_emb_dim))
        if self.hidden_dim is None:
            h = torch.tanh(self.hidden(e))
        else:
            h = e
        return self.output(h)

In [9]:
class TDNNClassifier:

    def __init__(self, num_classes, wsz, ksz, cap_emb_dim, n_hu, hidden_dim=None):
        super().__init__()
        self.num_classes = num_classes
        self.wsz = wsz
        self.cap_emb_dim = cap_emb_dim
        self.n_hu = n_hu
        self.ksz = ksz
        self.conv = nn.Conv1d(
            in_channels=wsz + cap_emb_dim,
            out_channels=n_hu,
            kernal_size=ksz,
            padding=(ksz-1)//2
        )
        if hidden_dim is not None:
            self.hidden = nn.Linear(n_hu, hidden_dim)
            self.output = nn.Linear(hidden_dim, num_classes)
        else:
            self.output = nn.Linear(n_hu, num_classes)
        for layer in [self.hidden, self.output] if hidden_dim else [self.output]:
            nn.init.uniform_(layer.weight, -0.05, 0.05)
            nn.init.uniform_(layer.bias)

    def forward(self, features):
        features = features.permute(0, 1, 2)
        conv_out = self.conv(features)
        conv_out = torch.tanh(conv_out)
        conv_out = conv_out.max(dim=2)
        if hidden_dim is None:
            conv_out = torch.tanh(self.hidden(conv_out))
        return self.output(conv_out)

In [12]:
lm_train = lm_dataset["train"]
lm_test = lm_dataset["test"]
lm_validation = lm_dataset["validation"]

In [23]:
"<Pad>"[1].isupper()

True

In [13]:
ner_train_df = pd.DataFrame(ner_dataset["train"])

In [17]:
vocab, idx_to_word, word_to_idx = build_vocab(lm_train)